# Fine-Tuned LoRA vs Baseline Evaluation

This notebook evaluates the original FLAN-T5 baselines against the PEFT LoRA fine-tuned model.

Compared systems:

- `pretrained`
- `prompt_engineered`
- `rag_system`
- `fine_tuned_lora`
- `rag_plus_lora`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/Lenovo/Desktop/sem 6/Gen_AI Project')

In [2]:
from src.baseline_eval import run_baseline_evaluation
import pandas as pd

pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 160)

## Configuration

In [3]:
SAMPLE_SIZE = 200
BASE_MODEL = 'google/flan-t5-base'
LORA_ADAPTER_PATH = PROJECT_ROOT / 'models' / 'flan_t5_lora'
FINETUNE_TEST_PATH = PROJECT_ROOT / 'data' / 'finetune_test.jsonl'
CHROMA_DIR = PROJECT_ROOT / 'data' / 'chroma'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_PREFIX = f'lora_comparison_{SAMPLE_SIZE}'

required_files = [
    LORA_ADAPTER_PATH / 'adapter_config.json',
    LORA_ADAPTER_PATH / 'adapter_model.safetensors',
    FINETUNE_TEST_PATH,
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))

print('Project root:', PROJECT_ROOT)
print('LoRA adapter:', LORA_ADAPTER_PATH)
print('Evaluation examples:', SAMPLE_SIZE)
print('Output prefix:', OUTPUT_PREFIX)

Project root: /Users/Lenovo/Desktop/sem 6/Gen_AI Project
LoRA adapter: /Users/Lenovo/Desktop/sem 6/Gen_AI Project/models/flan_t5_lora
Evaluation examples: 200
Output prefix: lora_comparison_200


## Run Evaluation

This cell generates outputs for all five strategies and writes the comparison files to `outputs/`.

In [4]:
metrics_df, aggregate_df, generations_df = run_baseline_evaluation(
    finetune_test_path=FINETUNE_TEST_PATH,
    chroma_dir=CHROMA_DIR,
    output_dir=OUTPUT_DIR,
    sample_size=SAMPLE_SIZE,
    model_name=BASE_MODEL,
    lora_adapter_path=LORA_ADAPTER_PATH,
    output_prefix=OUTPUT_PREFIX,
)

aggregate_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Evaluating example 1/200: paper_summary


KeyboardInterrupt: 

## Aggregate Comparison

In [ ]:
aggregate_df.round(4)

## Task-Level Comparison

In [ ]:
task_level_df = (
    metrics_df.groupby(['model', 'task'])[['bleu', 'rougeL', 'bertscore_f1']]
    .mean()
    .reset_index()
    .sort_values(['task', 'rougeL'], ascending=[True, False])
)
task_level_df.round(4)

## Sample Generations

In [ ]:
sample_generations = generations_df[
    generations_df['strategy'].isin(['pretrained', 'rag_system', 'fine_tuned_lora', 'rag_plus_lora'])
].head(12)

sample_generations[['example_id', 'strategy', 'task', 'topic', 'candidate']]

## Output Files

In [ ]:
output_files = [
    OUTPUT_DIR / f'{OUTPUT_PREFIX}_eval_data.csv',
    OUTPUT_DIR / f'{OUTPUT_PREFIX}_generations.csv',
    OUTPUT_DIR / f'{OUTPUT_PREFIX}_metrics.csv',
    OUTPUT_DIR / f'{OUTPUT_PREFIX}_comparison_table.csv',
    OUTPUT_DIR / f'{OUTPUT_PREFIX}_comparison.md',
]

for path in output_files:
    print(path)